# YouTube Toxic Comment Classification

## Part 1 — Introduction

### Student details

- IdoM — ID ending: [8349]
- ShirZ — ID ending: [4811]
- RoeeS — ID ending: [5498]

### AI prompts and additional resources

| Tool or resource | Prompt | Purpose |
|---|---|---|
| ChatGPT / Codex | "Take a look at the assignment and explain what is required." | Understanding the assignment requirements. |
| ChatGPT / Codex | "Which algorithm options are suitable for this classification problem?" | Selecting a learning algorithm. |
| ChatGPT / Codex | "Do we need to define quality metrics?" | Understanding the required evaluation metric. |
| Kaggle | https://www.kaggle.com/datasets/reihanenamdari/youtube-toxicity-data | Dataset source. |

### Learning problem and dataset

This project addresses a supervised binary text-classification problem: predicting whether an English YouTube comment is toxic. The input is the comment from the `Text` column, and the target is `IsToxic`, where `TRUE` represents a toxic comment and `FALSE` represents a non-toxic comment. The selected Kaggle dataset contains 1,000 manually labelled YouTube comments and additional labels describing different toxicity categories.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.20

### Loading the train and test sets

The selected Kaggle dataset contains one CSV file rather than predefined train and test files. Therefore, the data is divided once using a fixed, stratified 80/20 split. Exact duplicate comments are removed before the split so the same text cannot appear in both sets. The test set is kept separate and will be used only for the final evaluation.

In [ ]:
raw_df = pd.read_csv("youtoxic_english_1000.csv")

df = raw_df[["CommentId", "VideoId", "Text", "IsToxic"]].copy()
df["IsToxic"] = (
    df["IsToxic"]
    .astype(str)
    .str.upper()
    .map({"FALSE": 0, "TRUE": 1})
)

duplicate_key = (
    df["Text"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df = df.loc[~duplicate_key.duplicated()].reset_index(drop=True)

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df["IsToxic"],
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
train_df.head(5)

In [ ]:
test_df.head(5)

# Quality metric

This is a binary classification problem with one central class: toxic comments. Therefore, according to the assignment instructions, model quality will be evaluated using the F1 score for the toxic class only (`IsToxic = 1`).

The F1 score combines precision and recall. Precision measures how many of the comments predicted as toxic are actually toxic, while recall measures how many of the truly toxic comments were correctly identified. This is appropriate for the current task because both failing to detect a toxic comment and incorrectly flagging a non-toxic comment are important errors.

The F1 score is calculated as:

$$
F1 = 2 \cdot \frac{\mathrm{Precision} \cdot \mathrm{Recall}}{\mathrm{Precision} + \mathrm{Recall}}
$$

The same metric will be used throughout cross-validation, hyperparameter selection, and final test-set evaluation.

In [ ]:
from sklearn.metrics import f1_score

def calculate_quality(y_true, y_pred):
    """Calculate the F1 score for the toxic class."""
    return f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0,
    )

# Part 2 — Feature Engineering

## Text preprocessing

Machine-learning algorithms cannot process raw text directly. Therefore, the comments must first be normalized and tokenized.

The preprocessing includes conversion to lowercase, removal of HTML tags and URLs, and tokenization. Stop-word removal and Porter stemming are optional operations whose effect will later be evaluated using 5-fold cross-validation. Negation words such as `not`, `no`, and `never` are preserved because removing them may change the meaning of a sentence.

The original comments remain unchanged. Only the text supplied to the feature-extraction stage is transformed.

In [ ]:
import html
import re
import numpy as np

from nltk.stem import PorterStemmer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

porter_stemmer = PorterStemmer()
negation_words = {"no", "nor", "not", "never"}
safe_stop_words = set(ENGLISH_STOP_WORDS) - negation_words


class TextPreprocessor(BaseEstimator, TransformerMixin):
    """Clean and normalize English comments."""

    def __init__(self, remove_stopwords=False, use_stemming=False):
        self.remove_stopwords = remove_stopwords
        self.use_stemming = use_stemming

    def fit(self, X, y=None):
        return self

    def clean_comment(self, text):
        text = html.unescape(str(text)).lower()
        text = re.sub(r"<[^>]+>", " ", text)
        text = re.sub(r"https?://\S+|www\.\S+", " ", text)
        text = re.sub(r"\bcan't\b", "can not", text)
        text = re.sub(r"\bwon't\b", "will not", text)
        text = re.sub(r"n't\b", " not", text)
        tokens = re.findall(r"[a-z]+(?:'[a-z]+)?", text)

        if self.remove_stopwords:
            tokens = [token for token in tokens if token not in safe_stop_words]

        if self.use_stemming:
            tokens = [porter_stemmer.stem(token) for token in tokens]

        return " ".join(tokens)

    def transform(self, X):
        return [self.clean_comment(text) for text in X]

The preprocessing procedure is demonstrated on three comments from the training set and three comments from the test set. For each comment, basic preprocessing, stop-word removal, and Porter stemming are presented. The test examples are transformed only for demonstration and are not used to learn a vocabulary or select preprocessing parameters.

In [ ]:
basic_preprocessor = TextPreprocessor(
    remove_stopwords=False,
    use_stemming=False,
)

stopword_preprocessor = TextPreprocessor(
    remove_stopwords=True,
    use_stemming=False,
)

stemming_preprocessor = TextPreprocessor(
    remove_stopwords=True,
    use_stemming=True,
)

selected_train_indices = train_df.sample(n=3, random_state=42).index.tolist()
selected_test_indices = test_df.sample(n=3, random_state=42).index.tolist()

### Three preprocessing examples from the train and test sets

In [ ]:
def create_preprocessing_examples(source_df, row_indices, dataset_name):
    examples = source_df.loc[row_indices, ["Text", "IsToxic"]].copy()
    examples.insert(0, "Dataset", dataset_name)
    examples["Basic preprocessing"] = basic_preprocessor.transform(examples["Text"])
    examples["Without stop words"] = stopword_preprocessor.transform(examples["Text"])
    examples["With Porter stemming"] = stemming_preprocessor.transform(examples["Text"])
    return examples.reset_index(drop=True)


train_preprocessing_examples = create_preprocessing_examples(
    train_df, selected_train_indices, "Train"
)
test_preprocessing_examples = create_preprocessing_examples(
    test_df, selected_test_indices, "Test"
)

display(pd.concat(
    [train_preprocessing_examples, test_preprocessing_examples],
    ignore_index=True,
))

## TF-IDF feature extraction

After preprocessing, the comments are converted into numerical feature vectors using TF-IDF. Term Frequency measures how frequently a term occurs in a particular comment. Inverse Document Frequency reduces the weight of terms that occur in many training comments and increases the relative importance of less common terms.

$$\operatorname{TFIDF}(t,d)=\operatorname{TF}(t,d)\cdot\operatorname{IDF}(t)$$

The representation includes unigrams and bigrams. Unigrams represent individual words, while bigrams represent pairs of adjacent words and preserve limited local context. Sublinear term frequency is applied so repeated appearances of a term do not increase its influence linearly. L2 normalization scales every comment vector to unit length.

The vectorizer is fitted on the training set only. The test set is transformed using the vocabulary and IDF values learned from the training set.

In [ ]:
processed_train_text = basic_preprocessor.transform(train_df["Text"])
processed_test_text = basic_preprocessor.transform(test_df["Text"])

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    sublinear_tf=True,
    norm="l2",
)

X_train_tfidf = tfidf_vectorizer.fit_transform(processed_train_text)
X_test_tfidf = tfidf_vectorizer.transform(processed_test_text)

print("Train feature matrix shape:", X_train_tfidf.shape)
print("Test feature matrix shape:", X_test_tfidf.shape)

### TF-IDF examples

For each example, the following function displays the original comment and the eight features with the highest non-zero TF-IDF weights. Each feature represents either one word or a pair of adjacent words.

In [ ]:
def show_tfidf_examples(source_df, feature_matrix, vectorizer, row_indices, dataset_name, top_n=8):
    feature_names = vectorizer.get_feature_names_out()
    result_rows = []

    for row_index in row_indices:
        feature_row = feature_matrix.getrow(row_index)
        sorted_positions = np.argsort(feature_row.data)[::-1][:top_n]
        highest_features = []

        for position in sorted_positions:
            feature_index = feature_row.indices[position]
            feature_name = feature_names[feature_index]
            feature_value = feature_row.data[position]
            highest_features.append(f"{feature_name}: {feature_value:.3f}")

        result_rows.append({
            "Dataset": dataset_name,
            "IsToxic": source_df.loc[row_index, "IsToxic"],
            "Text": source_df.loc[row_index, "Text"],
            "Highest TF-IDF features": ", ".join(highest_features),
        })

    return pd.DataFrame(result_rows)

### Three train-set examples

In [ ]:
train_tfidf_examples = show_tfidf_examples(
    train_df,
    X_train_tfidf,
    tfidf_vectorizer,
    selected_train_indices,
    "Train",
)
display(train_tfidf_examples)

### Three test-set examples

In [ ]:
test_tfidf_examples = show_tfidf_examples(
    test_df,
    X_test_tfidf,
    tfidf_vectorizer,
    selected_test_indices,
    "Test",
)
display(test_tfidf_examples)

# Part 3 — Learning Algorithm Implementation

## Logistic Regression

Logistic regression is a supervised binary-classification algorithm. For every TF-IDF comment vector $x$, the model first calculates a linear score:

$$z = w^T x + b$$

The sigmoid function converts the score into a probability between 0 and 1:

$$P(y=1\mid x)=\sigma(z)=\frac{1}{1+e^{-z}}$$

Here, $P(y=1\mid x)$ is the estimated probability that a comment is toxic. A probability greater than or equal to the decision threshold is classified as toxic.

The parameters are learned by minimizing binary cross-entropy with L2 regularization:

$$J(w,b)=-\frac{1}{m}\sum_{i=1}^{m}\left[y_i\log(p_i)+(1-y_i)\log(1-p_i)\right]+\frac{1}{2Cm}\lVert w\rVert_2^2$$

Batch gradient descent repeatedly updates the weights and intercept in the direction that reduces this loss:

$$w \leftarrow w-\alpha\frac{\partial J}{\partial w},\qquad b \leftarrow b-\alpha\frac{\partial J}{\partial b}$$

The implementation below was written for this assignment and does not use `sklearn.linear_model.LogisticRegression`. It supports sparse TF-IDF matrices and provides separate `fit`, `predict_proba`, and `predict` methods.

### Hyperparameters

- `learning_rate` controls the size of each gradient-descent update. A value that is too small causes slow learning, while a value that is too large may prevent convergence.
- `max_iter` is the maximum number of gradient-descent iterations.
- `C` is the inverse L2-regularization strength. Smaller values produce stronger regularization.
- `threshold` converts the predicted probability into class 0 or 1.
- `tol` is the minimum loss improvement required to continue training.

Different values will later be evaluated using 5-fold cross-validation and the F1 score of the toxic class.

In [ ]:
import matplotlib.pyplot as plt
from scipy import sparse
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted


class CustomLogisticRegression(ClassifierMixin, BaseEstimator):
    """Binary logistic regression trained with batch gradient descent."""

    def __init__(
        self,
        learning_rate=0.5,
        max_iter=1000,
        C=1.0,
        threshold=0.5,
        tol=1e-7,
    ):
        self.learning_rate = learning_rate
        self.max_iter = max_iter
        self.C = C
        self.threshold = threshold
        self.tol = tol

    @staticmethod
    def _sigmoid(scores):
        scores = np.clip(scores, -500, 500)
        return 1.0 / (1.0 + np.exp(-scores))

    def _validate_hyperparameters(self):
        if self.learning_rate <= 0:
            raise ValueError("learning_rate must be positive.")
        if self.max_iter <= 0:
            raise ValueError("max_iter must be positive.")
        if self.C <= 0:
            raise ValueError("C must be positive.")
        if not 0 < self.threshold < 1:
            raise ValueError("threshold must be between 0 and 1.")
        if self.tol < 0:
            raise ValueError("tol cannot be negative.")

    def fit(self, X, y):
        """Learn the weights and intercept from labelled training examples."""
        self._validate_hyperparameters()
        X = sparse.csr_matrix(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64).reshape(-1)

        if X.shape[0] != y.shape[0]:
            raise ValueError("X and y must contain the same number of examples.")
        if not np.all(np.isin(y, [0.0, 1.0])):
            raise ValueError("Labels must be encoded as 0 and 1.")

        n_samples, n_features = X.shape
        self.n_features_in_ = n_features
        self.classes_ = np.array([0, 1])
        self.weights_ = np.zeros(n_features, dtype=np.float64)
        self.intercept_ = 0.0
        self.loss_history_ = []

        regularization_strength = 1.0 / (self.C * n_samples)
        previous_loss = np.inf

        for iteration in range(self.max_iter):
            scores = np.asarray(X @ self.weights_).reshape(-1) + self.intercept_
            probabilities = self._sigmoid(scores)
            errors = probabilities - y

            weight_gradient = np.asarray(X.T @ errors).reshape(-1) / n_samples
            weight_gradient += regularization_strength * self.weights_
            intercept_gradient = errors.mean()

            self.weights_ -= self.learning_rate * weight_gradient
            self.intercept_ -= self.learning_rate * intercept_gradient

            updated_scores = np.asarray(X @ self.weights_).reshape(-1) + self.intercept_
            data_loss = np.mean(
                np.logaddexp(0.0, updated_scores) - y * updated_scores
            )
            penalty = 0.5 * regularization_strength * np.dot(
                self.weights_, self.weights_
            )
            current_loss = data_loss + penalty
            self.loss_history_.append(current_loss)

            if abs(previous_loss - current_loss) < self.tol:
                break
            previous_loss = current_loss

        self.n_iter_ = iteration + 1
        self.coef_ = self.weights_.reshape(1, -1)
        return self

    def decision_function(self, X):
        """Return the linear score for each example."""
        check_is_fitted(self, ["weights_", "intercept_"])
        X = sparse.csr_matrix(X, dtype=np.float64)

        if X.shape[1] != self.n_features_in_:
            raise ValueError(
                "X has a different number of features than the training data."
            )

        return np.asarray(X @ self.weights_).reshape(-1) + self.intercept_

    def predict_proba(self, X):
        """Return probabilities for the non-toxic and toxic classes."""
        toxic_probability = self._sigmoid(self.decision_function(X))
        return np.column_stack([
            1.0 - toxic_probability,
            toxic_probability,
        ])

    def predict(self, X):
        """Predict 1 for toxic and 0 for non-toxic comments."""
        toxic_probability = self.predict_proba(X)[:, 1]
        return (toxic_probability >= self.threshold).astype(int)

### Implementation check on the training set

The next cell verifies that the implementation can train and predict, that all returned probabilities are valid, and that the loss decreases during optimization. This is only a technical check on the training set. It is not used to estimate final model quality or select hyperparameters, and the test set remains untouched.

In [ ]:
initial_model = CustomLogisticRegression(
    learning_rate=0.5,
    max_iter=1000,
    C=1.0,
    threshold=0.5,
    tol=1e-7,
)

initial_model.fit(
    X_train_tfidf,
    train_df["IsToxic"],
)

initial_train_probabilities = initial_model.predict_proba(X_train_tfidf)
initial_train_predictions = initial_model.predict(X_train_tfidf)

assert initial_train_probabilities.shape == (len(train_df), 2)
assert np.all((initial_train_probabilities >= 0) & (initial_train_probabilities <= 1))
assert np.allclose(initial_train_probabilities.sum(axis=1), 1.0)
assert set(np.unique(initial_train_predictions)).issubset({0, 1})
assert initial_model.loss_history_[-1] < initial_model.loss_history_[0]

print("Iterations performed:", initial_model.n_iter_)
print("Initial loss:", round(initial_model.loss_history_[0], 6))
print("Final loss:", round(initial_model.loss_history_[-1], 6))
print("Implementation checks passed successfully.")

In [ ]:
training_prediction_examples = train_df.loc[:4, ["Text", "IsToxic"]].copy()
training_prediction_examples["Toxic probability"] = (
    initial_train_probabilities[:5, 1].round(4)
)
training_prediction_examples["Prediction"] = initial_train_predictions[:5]
display(training_prediction_examples)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(initial_model.loss_history_, color="#4C78A8")
plt.xlabel("Gradient-descent iteration")
plt.ylabel("Regularized binary cross-entropy")
plt.title("Training-loss convergence of custom logistic regression")
plt.grid(alpha=0.25)
plt.show()

The decreasing loss shows that gradient descent updates the parameters in the intended direction. The model has not yet been evaluated on the test set. Hyperparameter and feature-engineering combinations will first be compared using cross-validation on the training set. The selected configuration will then be retrained on the complete training set.